# Module `visualization.py`

Ce notebook montre comment visualiser les graphes CESIPATH avec `matplotlib` [1]. Il couvre les trois vues du reseau : base, residuel et dynamique.

La visualisation n'est pas seulement decorative. Elle sert a verifier rapidement que les invariants racontes dans les autres notebooks correspondent a une forme lisible : le graphe de base est connecte, les interdictions ne cassent pas le residuel, les surcouts sont visibles, et la dynamique ne vide pas le reseau.

Attention aux boutons dans Jupyter : selon le backend Jupyter/matplotlib, le bouton `->` peut etre statique [2, 3]. Le notebook fournit donc aussi une methode de fallback en appelant `advance_session` directement.


In [ ]:
%matplotlib inline

In [ ]:
import sys
from pathlib import Path

sys.path.append(str(Path.cwd().parent / "src"))

from cesipath.graph_generator import GraphGenerator
from cesipath.models import GraphGenerationConfig
from cesipath.visualization import GraphVisualizer

## 1. Semantique des couleurs

Constantes exposees par `GraphVisualizer` :

| Constante | Couleur | Signification |
|---|---|---|
| `BASE_EDGE_COLOR` | `#7a7a7a` gris | Route de base, avant contraintes |
| `ACTIVE_EDGE_COLOR` | `#2a9d8f` vert | Route libre ou active |
| `SURCHARGED_EDGE_COLOR` | `#7b2cbf` violet | Route surchargee statiquement |
| `FORBIDDEN_EDGE_COLOR` | `#c1121f` rouge | Route interdite statiquement |
| `TEMP_DISABLED_EDGE_COLOR` | `#f4a261` orange | Route temporairement OFF en dynamique |
| `DEPOT_COLOR` | `#264653` bleu fonce | Depot |
| `CLIENT_COLOR` | `#8ecae6` bleu clair | Client |

La couleur encode le statut, tandis que le style de ligne (`-` ou `--`) distingue les routes utilisables des routes indisponibles. Les labels d'aretes affichent le cout correspondant a la vue : cout de base, cout statique ou cout dynamique.


## 2. Generation d'une instance a visualiser

On prend une taille modeste (8 sommets) pour que les labels de couts restent lisibles. Sur des graphes plus grands, la visualisation fonctionne encore, mais les etiquettes se chevauchent rapidement.

Les parametres dynamiques sont volontairement visibles dans la configuration : ils permettent de produire des changements en quelques clics sans rendre le reseau instable.


In [ ]:
config = GraphGenerationConfig(
    node_count=8,
    seed=42,
    auto_density_profile=True,
    forbidden_rate=0.1,
    surcharge_rate=0.25,
    dynamic_sigma=0.18,
    dynamic_mean_reversion_strength=0.35,
    dynamic_max_multiplier=1.8,
    dynamic_forbid_probability=0.05,
    dynamic_restore_probability=0.2,
    dynamic_max_disabled_ratio=0.2,
)

generator = GraphGenerator(config)
instance = generator.generate()
visualizer = GraphVisualizer(instance, generator)
instance.summary()

## 3. `show_base_graph` - graphe physique brut

Affiche uniquement les vraies routes physiques avec leur cout de base. Aucune contrainte statique n'est appliquee a ce stade.

Cette vue repond a la question : **quelles routes existent dans le monde avant travaux, interdictions et surcouts ?** Elle sert de reference pour comprendre ce que les etapes suivantes retirent ou modifient.

Les aretes sont grises, les sommets sont places selon leurs coordonnees, et le depot a une couleur distincte pour faciliter la lecture des tournees futures.


In [ ]:
visualizer.show_base_graph()

## 4. `show_residual_graph` - apres contraintes statiques

Affiche le graphe apres application des interdictions et surcouts statiques. Cette vue montre exactement ce que Dijkstra voit pour construire la matrice metrique statique.

Interpretation :

- vert : route utilisable au cout statique normal ;
- violet : route utilisable mais surchargee ;
- rouge pointille : route interdite, donc ignoree dans la fermeture metrique.

C'est souvent la vue la plus utile pour debugger le generateur. Si le residuel semble trop pauvre ou trop dense, il faut regarder les parametres de densite, `forbidden_rate`, `surcharge_rate` et les seuils du validateur.


In [ ]:
visualizer.show_residual_graph()

## 5. `show_dynamic_graph` - session interactive

Retourne un `GraphVisualizationSession` qui contient :

- le simulateur dynamique ;
- le snapshot courant ;
- la figure matplotlib ;
- l'axe de dessin ;
- le bouton `->` (`matplotlib.widgets.Button` [2]).

Le titre de la figure affiche en temps reel : numero de tour, nombre d'aretes actives, densite active et ratio OFF. Ce sont les memes grandeurs que `DynamicStateValidator` surveille.

La session est un objet plutot qu'une simple figure parce qu'un clic doit mettre a jour plusieurs choses a la fois : avancer le simulateur, remplacer le snapshot, redessiner les aretes et rafraichir le canvas.


In [ ]:
session = visualizer.show_dynamic_graph()
session.fig

## 6. Fallback : avancer manuellement

Si le bouton `->` ne reagit pas dans votre frontend Jupyter (par exemple avec `%matplotlib inline`), la methode `advance_session` fait la meme chose en code.

Ce fallback est aussi pratique pour les tests pedagogiques : on peut avancer de plusieurs tours dans une boucle, inspecter `session.snapshot`, puis afficher la figure finale.

Le comportement est identique au clic : le simulateur avance d'un tour, la figure est redessinee, et la session garde le nouveau snapshot.


In [ ]:
visualizer.advance_session(session)
session.fig

## 7. Visualisation d'une solution de tournee

Le sous-module `cesipath.algorithms.visualization` expose deux fonctions utiles :

- `plot_solution(instance, solution)` : trace les tournees sur le graphe, une couleur par vehicule, sur fond du graphe residuel ;
- `save_solution_plot(...)` : sauvegarde la figure avec un index auto-incremente.

Cette visualisation est differente de `GraphVisualizer`. Ici, l'objectif n'est pas de montrer tous les statuts du reseau, mais de lire une solution VRP : quels clients sont servis ensemble, dans quel ordre, et combien de tournees sont utilisees.

Une solution est calculee sur la matrice complete. Le trace aide donc a verifier si les routes abstraites restent plausibles geographiquement sur l'instance.


In [ ]:
from cesipath.algorithms import grasp, plot_solution
from cesipath.solver_input import build_static_solver_input

solver_input = build_static_solver_input(instance)
solution = grasp(solver_input, max_iterations=5, rcl_alpha=0.2, seed=42)
plot_solution(instance, solution)

## 8. Pourquoi pas d'accents dans les labels ?

Le projet suit une convention historique : commentaires et chaines en francais **sans accents**. Cela evite les problemes d'encodage sur les vieilles consoles ou les environnements mal configures.

Dans les figures matplotlib, cette convention a aussi un avantage pratique : les labels restent simples a rendre quel que soit le backend. Ce n'est pas indispensable dans un environnement moderne, mais cela garde les notebooks homogenes avec le reste du code.

A retenir : `visualization.py` est un outil d'inspection. Il permet de voir les hypotheses du modele, pas seulement de produire une image jolie.

---

## References

[1] **Hunter, J. D. (2007).** *Matplotlib: A 2D graphics environment.* Computing in Science & Engineering, 9(3), 90-95. https://doi.org/10.1109/MCSE.2007.55

[2] **Matplotlib Development Team.** *matplotlib.widgets.Button.* Matplotlib documentation. https://matplotlib.org/stable/api/widgets_api.html#matplotlib.widgets.Button

[3] **Project Jupyter.** *Jupyter Notebook documentation.* https://jupyter-notebook.readthedocs.io/
